# Fase 3 — Limpieza de Datos
**Proyecto:** Plato's Pizza Sales 2015  
**Fuente raw:** `data/raw/`  
**Output limpio:** `data/clean/pizza_data_clean.csv`

Problemas identificados en Fase 2 a resolver aquí:
1. Encoding cp1252 (Windows) — los CSVs no son UTF-8
2. Líneas vacías al final de `pizzas.csv` y `pizza_types.csv`
3. Tipos de dato: `date` y `time` llegan como string → convertir a datetime
4. Encoding roto en `calabrese` ingredientes
5. Verificar nulls, duplicados y rango de `quantity`
6. Verificar integridad referencial entre tablas (foreign keys)
7. Construir tabla maestra unificada con JOIN de las 4 tablas

In [ ]:
import pandas as pd
import numpy as np
import re

RAW   = '../data/raw/'
CLEAN = '../data/clean/'

# Los CSVs originales están en encoding cp1252 (Windows) — no UTF-8
orders        = pd.read_csv(RAW + 'orders.csv',        encoding='cp1252')
order_details = pd.read_csv(RAW + 'order_details.csv', encoding='cp1252')
pizzas        = pd.read_csv(RAW + 'pizzas.csv',        encoding='cp1252')
pizza_types   = pd.read_csv(RAW + 'pizza_types.csv',   encoding='cp1252')

print('orders:       ', orders.shape)        # (21350, 3)
print('order_details:', order_details.shape) # (48620, 4)
print('pizzas:       ', pizzas.shape)        # (96, 4)
print('pizza_types:  ', pizza_types.shape)   # (32, 4)

## 3.1 — Nulls por tabla
**Resultado real:** 0 nulls en todas las tablas. Dataset muy limpio.

In [ ]:
for name, df in [('orders', orders), ('order_details', order_details),
                 ('pizzas', pizzas), ('pizza_types', pizza_types)]:
    nulls = df.isnull().sum()
    has_nulls = nulls[nulls > 0]
    print(f'{name}: {has_nulls.to_dict() if len(has_nulls) else "Sin nulls"}')

# Eliminar filas completamente vacías (líneas en blanco al final del CSV)
# Nota: en este dataset no había líneas vacías reales, pero el dropna es preventivo
pizzas      = pizzas.dropna(how='all').reset_index(drop=True)
pizza_types = pizza_types.dropna(how='all').reset_index(drop=True)
print('\nTras dropna - pizzas:', pizzas.shape, '| pizza_types:', pizza_types.shape)

## 3.2 — Tipos de dato

In [ ]:
print('Tipos ANTES:')
print(orders.dtypes)

orders['date'] = pd.to_datetime(orders['date'])
orders['time'] = pd.to_datetime(orders['time'], format='%H:%M:%S').dt.time

print('\nTipos DESPUES:')
print(orders.dtypes)

# price ya es float64 — no requiere conversión
print('\nPrice dtype:', pizzas['price'].dtype)
print(pizzas['price'].describe().round(2))

## 3.3 — Duplicados
**Resultado real:** 0 duplicados en todas las tablas.

In [ ]:
for name, df in [('orders', orders), ('order_details', order_details),
                 ('pizzas', pizzas), ('pizza_types', pizza_types)]:
    print(f'{name}: {df.duplicated().sum()} duplicados exactos')

## 3.4 — Rango y distribución de `quantity`
**Resultado real:** min=1, max=4. El 98% de las líneas tienen quantity=1.

In [ ]:
print(order_details['quantity'].describe())
print('\nDistribucion:')
print(order_details['quantity'].value_counts().sort_index())
# quantity=1: 47,693 | quantity=2: 903 | quantity=3: 21 | quantity=4: 3
# Total pizzas vendidas = SUM(quantity) = 49,574  (no 48,620)

## 3.5 — Integridad referencial (Foreign Keys)
**Resultado real:** 0 huérfanos en todas las relaciones. Integridad perfecta.

In [ ]:
orphan_orders = order_details[~order_details['order_id'].isin(orders['order_id'])]
print('order_details sin order_id valido:', len(orphan_orders))

orphan_pizzas = order_details[~order_details['pizza_id'].isin(pizzas['pizza_id'])]
print('order_details sin pizza_id valido:', len(orphan_pizzas))

orphan_types  = pizzas[~pizzas['pizza_type_id'].isin(pizza_types['pizza_type_id'])]
print('pizzas sin pizza_type_id valido:  ', len(orphan_types))

## 3.6 — Corrección de encoding en ingredients
El carácter `\x91` (cp1252) en `calabrese` se reemplaza por apóstrofo ASCII.

In [ ]:
print('Antes repr:', repr(pizza_types[pizza_types['pizza_type_id'] == 'calabrese']['ingredients'].values[0][:30]))

pizza_types['ingredients'] = pizza_types['ingredients'].apply(
    lambda x: re.sub(r'[\x80-\x9f\u0080-\u009f\u2018\u2019]Nduja', "'Nduja", x) if isinstance(x, str) else x
)

print('Despues:   ', pizza_types[pizza_types['pizza_type_id'] == 'calabrese']['ingredients'].values[0][:40])

## 3.7 — Construcción de tabla maestra (JOIN)

In [ ]:
df = (order_details
      .merge(orders,      on='order_id',     how='left')
      .merge(pizzas,      on='pizza_id',      how='left')
      .merge(pizza_types, on='pizza_type_id', how='left')
)

df['revenue'] = df['price'] * df['quantity']

# Sanity check: el JOIN no debe crear ni perder filas
assert len(df) == len(order_details), f'ERROR: {len(df)} != {len(order_details)}'
print('Sanity check OK - filas:', len(df))
print('Columnas:', df.columns.tolist())

print('\nNulls en columnas clave post-JOIN:')
key_cols = ['order_id', 'pizza_id', 'date', 'price', 'category', 'name']
print(df[key_cols].isnull().sum())

print('\nRevenue total 2015: $' + f'{df["revenue"].sum():,.2f}')  # $817,860.05
print('Total pizzas vendidas:', df['quantity'].sum())              # 49,574

df.head(3)

## 3.8 — Guardar dataset limpio

In [ ]:
df.to_csv(CLEAN + 'pizza_data_clean.csv',   index=False, encoding='utf-8')
orders.to_csv(CLEAN + 'orders_clean.csv',   index=False, encoding='utf-8')
pizza_types.to_csv(CLEAN + 'pizza_types_clean.csv', index=False, encoding='utf-8')

print('Archivos guardados en data/clean/')
print(f'pizza_data_clean.csv: {df.shape[0]:,} filas x {df.shape[1]} columnas')

## Resumen de decisiones de limpieza

| Problema | Acción tomada | Justificación |
|----------|--------------|---------------|
| Encoding cp1252 | `read_csv(encoding='cp1252')` | CSV exportado en Windows |
| Líneas vacías | `dropna(how='all')` | Artefacto del CSV, no son datos |
| `date` como string | `pd.to_datetime()` | Necesario para análisis temporal |
| `time` como string | `pd.to_datetime().dt.time` | Para análisis de horas pico |
| Encoding `calabrese` | `re.sub()` con regex | Error de exportación del CSV original |
| `brie_carre` solo talla S | Sin cambio | Característica del menú, no error |
| `the_greek` con XL/XXL | Sin cambio | Característica del menú, no error |

**Números clave validados:**
- Revenue total 2015: **$817,860.05**
- Total pizzas vendidas: **49,574** (≠ 48,620 líneas — porque quantity puede ser > 1)
- 0 nulls · 0 duplicados · 0 huérfanos en foreign keys